# 02 - Build Face Database

## Creating Your Known Faces Database

This notebook will guide you through:

1. 📸 Understanding the Face Database
2. 📁 Organizing Your Photos
3. 🔨 Building the Database
4. ✅ Verifying Database Contents
5. 👤 Adding Sample Person (Demo)

---

## 1. Understanding the Face Database

The face database stores images of known people. The system uses these to recognize faces in the video feed.

### Directory Structure

```
data/known_faces/
├── person1_name/
│   ├── photo1.jpg
│   ├── photo2.jpg
│   └── photo3.jpg
├── person2_name/
│   ├── photo1.jpg
│   └── photo2.jpg
└── ...
```

### Requirements
- **Minimum 3 photos per person**
- **Various angles and lighting**
- **Clear, unobstructed face**
- **Supported formats**: JPG, PNG

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from src.database_manager import DatabaseManager
from config import DATABASE_CONFIG
import matplotlib.pyplot as plt
import cv2
import numpy as np

print("Face Database Builder")
print("="*60)
print(f"Known faces directory: {DATABASE_CONFIG['known_faces_dir']}")
print(f"Database file: {DATABASE_CONFIG['encodings_file']}")
print(f"Minimum images per person: {DATABASE_CONFIG['min_images_per_person']}")
print("="*60)

## 2. Check Current Database Structure

In [ ]:
# Initialize database manager
db_manager = DatabaseManager(
    known_faces_dir=DATABASE_CONFIG['known_faces_dir'],
    encodings_file=DATABASE_CONFIG['encodings_file'],
    min_images_per_person=DATABASE_CONFIG['min_images_per_person'],
    max_images_per_person=DATABASE_CONFIG['max_images_per_person']
)

# Check what's currently in the directory
known_faces_dir = Path(DATABASE_CONFIG['known_faces_dir'])
person_dirs = [d for d in known_faces_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]

print(f"Found {len(person_dirs)} person directories:\n")

if person_dirs:
    for person_dir in person_dirs:
        images = list(person_dir.glob('*.jpg')) + list(person_dir.glob('*.png'))
        status = "✅" if len(images) >= DATABASE_CONFIG['min_images_per_person'] else "❌"
        print(f"{status} {person_dir.name}: {len(images)} images")
else:
    print("No person directories found yet.")
    print("\nTo add people, create subdirectories in data/known_faces/")
    print("Example: data/known_faces/john_doe/")

## 3. Add Photos Using Webcam (Optional)

You can capture photos directly from your webcam to add to the database.

In [ ]:
# This is a simplified version for Jupyter
# For full functionality, use: scripts/add_new_person.py

from src.camera_handler import CameraHandler
from config import CAMERA_CONFIG
import time

def capture_photos_simple(person_name, num_photos=5):
    """Capture photos from webcam for database."""
    print(f"\nCapturing {num_photos} photos of {person_name}")
    print("Position your face clearly and press 's' to capture each photo")
    print("Press 'q' to quit\n")
    
    # Create directory for person
    person_dir = DATABASE_CONFIG['known_faces_dir'] / person_name
    person_dir.mkdir(exist_ok=True)
    
    camera = CameraHandler(
        camera_source=CAMERA_CONFIG['source'],
        width=CAMERA_CONFIG['width'],
        height=CAMERA_CONFIG['height']
    )
    
    if not camera.open():
        print("❌ Could not open camera")
        return 0
    
    captured = 0
    
    while captured < num_photos:
        ret, frame = camera.read_frame()
        if not ret:
            break
        
        # Display frame
        display_frame = frame.copy()
        cv2.putText(display_frame, f"Photos: {captured}/{num_photos}", 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(display_frame, "Press 's' to capture", 
                   (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        cv2.imshow("Capture Photos", display_frame)
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord('s'):
            # Save photo
            photo_path = person_dir / f"photo_{captured+1}.jpg"
            cv2.imwrite(str(photo_path), frame)
            captured += 1
            print(f"✅ Captured photo {captured}/{num_photos}")
            time.sleep(0.5)
        
        elif key == ord('q'):
            break
    
    camera.release()
    cv2.destroyAllWindows()
    
    return captured

# Example usage (uncomment to use):
# person_name = input("Enter person name: ")
# num_captured = capture_photos_simple(person_name, 5)
# print(f"\nCaptured {num_captured} photos for {person_name}")

print("💡 Tip: For easier photo capture, use the command line script:")
print("   python scripts/add_new_person.py")

## 4. Build the Database

Once you've added photos for all people, build the database.

In [ ]:
print("Building face database...\n")

# Build database from known_faces directory
database = db_manager.build_database()

if database:
    print(f"\n✅ Database built successfully!")
    print(f"   Found {len(database)} people\n")
    
    # Save database
    if db_manager.save_database(database):
        print("✅ Database saved successfully!\n")
    else:
        print("❌ Failed to save database\n")
else:
    print("❌ No valid people found in database")
    print(f"   Make sure each person has at least {DATABASE_CONFIG['min_images_per_person']} photos")

## 5. Verify Database Contents

In [ ]:
# Load and display database statistics
stats = db_manager.get_database_stats()

print("Database Statistics")
print("="*60)
print(f"Total people: {stats['num_people']}")
print(f"Total images: {stats['total_images']}")
print("\nPeople in database:")

for name, info in stats['people'].items():
    print(f"  {name}: {info['num_images']} images")

print("="*60)

# Validate database
issues = db_manager.validate_database()
if issues:
    print("\n⚠️ Validation issues:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("\n✅ Database validation passed!")

## 6. Preview Database Images

In [ ]:
# Display sample images from each person
database = db_manager.load_database()

if database:
    for person_name, images in database.items():
        print(f"\n{person_name} - {len(images)} images")
        
        # Display first 3 images
        num_display = min(3, len(images))
        fig, axes = plt.subplots(1, num_display, figsize=(15, 5))
        
        if num_display == 1:
            axes = [axes]
        
        for idx in range(num_display):
            img_rgb = cv2.cvtColor(images[idx], cv2.COLOR_BGR2RGB)
            axes[idx].imshow(img_rgb)
            axes[idx].set_title(f"Photo {idx+1}")
            axes[idx].axis('off')
        
        plt.suptitle(person_name, fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("No database loaded yet")

## Summary

### ✅ Database Building Checklist

- [ ] Created directories for each person in `data/known_faces/`
- [ ] Added at least 3 photos per person
- [ ] Built the database successfully
- [ ] Verified database contents
- [ ] Database validation passed

### �� Next Steps

1. **Test Recognition**: Open `03_test_recognition.ipynb`
2. **Run Security System**: Open `04_run_security_system.ipynb`

### 💡 Tips

- Use photos with different angles and lighting
- Ensure faces are clearly visible
- Add more photos for better accuracy
- Regularly update the database

---

**Ready to test recognition?** → Open `03_test_recognition.ipynb`